# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pr120107/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

print("Working directory:", os.getcwd())

Working directory: /content/flyrank-ml-internship-starter


In [2]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET (
    TYPE huggingface,
    TOKEN '{HF_TOKEN}'
);
""")

rel = "hf://datasets/FlyRank/internship-warehouse"

print("Connected to FlyRank warehouse.")

Connected to FlyRank warehouse.


In [3]:
clients = con.sql(f"""
SELECT COUNT(*)
FROM read_parquet('{rel}/dim_clients.parquet')
""").fetchone()[0]

print(f"Clients in warehouse: {clients}")

Clients in warehouse: 104


In [4]:
sample = con.sql(f"""
SELECT *
FROM read_parquet(
    '{rel}/fact_content_daily_performance_sample.parquet'
)
LIMIT 5
""").df()

sample

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_3ffa76342f366962,content_1a6296faee432dae,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-01,client_3ffa76342f366962,content_73f21e612565035a,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-01,client_3ffa76342f366962,content_5a5be514ff559598,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-01,client_3ffa76342f366962,content_05b377d0c8a5cfd8,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-01,client_3ffa76342f366962,content_dc34c661d63e55a9,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


### **Unit of Analysis**

One row represents the daily search and analytics performance of one content item for one client on one report date.

### **Time Window**

This notebook uses a mid-panel month (2026-03) when analysing the warehouse. A mid-panel month is used because the final month (2026-06) is reserved as a natural outcome period and should not be used to develop label logic.

In [5]:
grain_check = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS duplicate_rows
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 10;
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,duplicate_rows


The query returned no duplicate rows, confirming that each row represents one content item for one client on one report date.

# Feature

The following fields are used as input features because they describe the search and engagement performance of a content item and are available at the decision moment.

- `gsc_impressions`
- `gsc_clicks`
- `gsc_avg_position`
- `ga4_sessions`
- `scroll_events`

---

# Label / Proxy

This lane uses **unsupervised clustering**, so there is no prediction label or target variable. The objective is to discover natural groups of similar content items rather than predict an outcome.

---

# Context

These fields identify observations or provide grouping information but are not used as model features.

- `report_date`
- `client_hash_id`
- `content_hash_id`
- `month`

---

# Excluded

The following fields are deliberately excluded from clustering.

- `client_has_gsc` - Indicates whether the client has Google Search Console connected and is used for context only.
- `client_has_ga4` - Indicates whether the client has Google Analytics connected and is used for context only.
- `gsc_data_available` - Used only to filter rows with valid Google Search Console data.
- `ga4_data_available` - Used only to filter rows with valid Google Analytics data.
- Any future or label-derived fields - Excluded to prevent data leakage.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [6]:
summary = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03';
""").df()

summary

,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


This query confirms the number of rows in the March 2026 slice and verifies the report date range used in this notebook.

In [7]:
availability = con.sql(f"""
SELECT
    COUNT(*) AS rows_after_filter
FROM read_parquet(
    '{rel}/fact_content_daily_performance/**/*.parquet'
)
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE;
""").df()

availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows_after_filter
0,3611061


This query counts the rows with Google Search Console data available. Filtering with IS TRUE ensures that only rows containing valid search data are included in the analysis.

# Data Limits

This analysis has several limitations.

- Different clients have different amounts of historical data because they connected Google Search Console and Google Analytics at different times.
- Rows where `gsc_data_available` or `ga4_data_available` are `FALSE` represent unavailable data rather than zero performance.
- This notebook only analyzes one mid-panel month (2026-03), so it cannot capture long-term trends or seasonal effects.
- The results describe observed patterns that support content review decisions and should not be interpreted as evidence of causal relationships.

In [8]:
client_history = con.sql(f"""
SELECT
    MIN(gsc_data_start) AS earliest_gsc_start,
    MAX(gsc_data_start) AS latest_gsc_start,
    MIN(ga4_data_start) AS earliest_ga4_start,
    MAX(ga4_data_start) AS latest_ga4_start
FROM read_parquet('{rel}/dim_clients.parquet');
""").df()

client_history

,earliest_gsc_start,latest_gsc_start,earliest_ga4_start,latest_ga4_start
0,2025-01-27,2026-06-02,2025-10-29,2026-06-01


The different Google Search Console and Google Analytics start dates confirm that clients have different history lengths. This means historical coverage is not balanced across all clients.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.